# 1.Load Data

In [ ]:
import numpy as np
import pickle
import anndata as an
import scanpy as sc
import pandas as pd
import os
from scipy.io import loadmat
import scipy
import mlx.core as mx
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
n_protein = 13
n_lipid = 664
n_slice = 25

In [ ]:
path = '/Volumes/ryxx_lyx/Mouse/MSI/'
dfs = []
lipid_data = []
protein_data = []
label_df = pd.read_csv(path+'label2superlabel_20200501.csv')
for dir in sorted(os.listdir(path)):
    if 'Mouse' in dir:
        dfs.append(
            pd.read_excel(
                path+dir+f'/{dir}_nuancename.xlsx'
            )
        )
        lipid_data.append(
            loadmat(path+dir+'/reduced_msi.mat')
        )
        protein_data.append(
            loadmat(path+dir+'/cellmarker_compen_rescale_-1_v2_intergration_148_t=3cosine_kmedoids_k=70-pxmslabel.mat')
        )
protein_name_data = loadmat('/Volumes/ryxx_lyx/Mouse/mat/cellmarker_compen_rescale_intergration_148.mat')
super_label = pd.read_csv('/Volumes/ryxx_lyx/Mouse/MSI/label2superlabel_20200501.csv')
print(dfs[0])
print(lipid_data[0].keys())
print(protein_data[0].keys())
print(protein_name_data.keys())

In [ ]:
spatials = []
lipids = []
proteins = []
lipid_names = lipid_data[0]['mz'].flatten()
lipid_names = [str(x) for x in lipid_names]
protein_names = [x.item() for x in protein_name_data['pic'][0]]
for data in lipid_data:
    spatials.append(data['xy_positions_new'])
    lipids.append(data['reduced_msi'].T)
for data in protein_data:
    proteins.append(data['nuancemat'])
print(lipids[0].shape)
print(proteins[0].shape)

In [ ]:
print(lipid_names[:10])
print(protein_names)

In [ ]:
for i in range(len(dfs)):
    dfs[i]['spatial'] = list(spatials[i])
dfs[0]

In [ ]:
label_df['finallabel'] = label_df.index
for i in range(len(dfs)):
    dfs[i] = dfs[i][dfs[i]['nuancename'] != 'No']
    #dfs[i]['finallabel'] = protein_data[i]['finallabel']
    #df = dfs[i][dfs[i]['finallabel'] != -1]
    #dfs[i] = df
    #dfs[i] = df.merge(label_df,on='finallabel',how='inner').drop(columns=['finallabel'])
dfs[0]

In [ ]:
spatial_plaques= []
for df in dfs:
    spatial_plaques.append(df.index.to_numpy())
spatial_plaques[0]

In [ ]:
for i in range(len(spatial_plaques)):
    proteins[i] = proteins[i][spatial_plaques[i]]
    lipids[i] = lipids[i][spatial_plaques[i]]
print(lipids[0].shape)
print(proteins[0].shape)

In [ ]:
np.stack(dfs[0]['spatial'])

In [ ]:
xys = []
for df in dfs:
    xys.append(np.stack(df['spatial']))

In [ ]:
with open('data/spatial/xys.pickle','wb') as f:
    pickle.dump(xys,f)

In [ ]:
with open('data/spatial/protein_name.pickle','wb') as f:
    pickle.dump(protein_names,f)
with open('data/spatial/lipid_name.pickle','wb') as f:
    pickle.dump(lipid_names,f)

In [ ]:
with open('data/spatial/protein_data.pickle','wb') as f:
    pickle.dump(proteins,f)
with open('data/spatial/lipid_data.pickle','wb') as f:
    pickle.dump(lipids,f)

# 2.Compute Correlation

## Pearson

In [ ]:
from scipy.special import betainc
def pearsonr2(x, y):

    n = x.shape[-1]
    xm = x.mean(axis=-1, keepdims=True)
    ym = y.mean(axis=-1, keepdims=True)
    cov = np.sum((x - xm) * (y - ym), axis=-1)/(n-1)
    sx = np.std(x, ddof=1, axis=-1)
    sy = np.std(y, ddof=1, axis=-1)
    rho = cov/(sx * sy)
    ab = n/2 - 1
    x = (abs(rho) + 1)/2
    p = 2*(1-betainc(ab, ab, x))
    return rho, p
def compute_pearson(X):
    
    corrs=np.ones((X.shape[1],X.shape[1]))
    p_vals=np.ones((X.shape[1],X.shape[1]))
    for i in range(X.shape[1]):
        for j in range(X.shape[1]):
            corr,p_val = pearsonr2(X[:,i],X[:,j])
            corrs[i][j] = corr
            p_vals[i][j] = p_val
    return corrs,p_vals

## Protein Adjacency Matrix

In [ ]:
from tqdm import tqdm
corr_protein = []
p_protein = []
corr_lipid = []
p_lipid = []
for i in tqdm(range(len(spatial_plaques))):
    c_p,p_p = compute_pearson(proteins[i])
    #c_l,p_l = compute_pearson(lipids[i])
    corr_protein.append(c_p)
    p_protein.append(p_p)
    #corr_lipid.append(c_l)
    #p_lipid.append(p_l)

In [ ]:
corr_protein[0].shape

In [ ]:
corr_protein = np.array(corr_protein)
p_protein = np.array(p_protein)
for i in range(n_protein):
    for j in range(n_protein):
        p_protein[:,i,j] = scipy.stats.false_discovery_control(p_protein[:,i,j]) 

In [ ]:
significant = (p_protein <= 0.05)
significant_counts = np.sum(significant, axis=0)
np.fill_diagonal(significant_counts, 0)
significant_counts

In [ ]:
avg = np.sum(significant_counts)/(n_protein**2-n_protein)
avg = np.ceil(avg)
avg

In [ ]:
p_weight = np.mean(corr_protein,axis=0)
p_weight = np.where(p_weight >0,p_weight,0)
p_adj = np.where(significant_counts >=avg, p_weight, 0)
p_adj

In [ ]:
plt.figure(figsize=(4, 6))
flatten_p_weights = p_adj.flatten()
#sns.kdeplot(flatten_p_weights[flatten_p_weights>0], fill=True)
sns.histplot(flatten_p_weights[flatten_p_weights>0])
plt.title('Box plot of weights for proteins')
plt.xlabel('weights', fontsize=12)
plt.ylabel('numbers', fontsize=12)
plt.show()

## Lipid Adjacency Matrix

In [ ]:
for i in tqdm(range(len(spatial_plaques))):
    c_l,p_l = compute_pearson(lipids[i])
    corr_lipid.append(c_l)
    p_lipid.append(p_l)
    
corr_lipid = np.array(corr_lipid)
p_lipid = np.array(p_lipid)

for i in range(n_lipid):
    for j in range(n_lipid):
        p_lipid[:,i,j] = scipy.stats.false_discovery_control(p_lipid[:,i,j])

In [ ]:
significant = (p_lipid <= 0.05)
significant_counts = np.sum(significant, axis=0)
np.fill_diagonal(significant_counts, 0)
significant_counts

In [ ]:
avg = np.sum(significant_counts)/(n_lipid**2-n_lipid)
avg = np.ceil(avg)
avg

In [ ]:
l_weight = np.mean(corr_lipid,axis=0)
l_weight = np.where(l_weight >0,l_weight,0)
l_adj = np.where(significant_counts >=avg, l_weight, 0)
l_adj

In [ ]:
plt.figure(figsize=(4, 6))
flatten_p_weights =l_adj.flatten()
#sns.kdeplot(flatten_p_weights[flatten_p_weights>0], fill=True)
sns.histplot(flatten_p_weights[flatten_p_weights>0])
plt.title('Box plot of weights for lipids')
plt.xlabel('All Data', fontsize=12)
plt.ylabel('lipid weights', fontsize=12)
plt.show()

## Moran

In [ ]:
def moranR_weights(pos,l=0.65):
    
    pos = mx.array(pos,dtype=mx.int32)
    diff = pos[:, mx.newaxis, :] - pos[mx.newaxis, :, :]
    d = mx.sum(diff**2, axis=-1) 
    d = mx.sqrt(d)
    

    n = len(pos)
    w = mx.exp(-d**2/(2 * l**2))
    W = mx.sum(w)

    weight = (n/W)*w
    #weight = w
    return weight
def moranR(x,y,w):
    l_s,l_x = x.shape
    l_y = y.shape[1]
    n = w.shape[0]
    
    x=mx.array(x).T
    y=mx.array(y).T
    
    x_mean = mx.mean(x, axis=1,keepdims=True)
    y_mean = mx.mean(y, axis=1,keepdims=True)

    x_diff = (x-x_mean) #/mx.std(x, axis=0, keepdims=True) #[:,mx.newaxis]
    y_diff = (y-y_mean) #/mx.std(y, axis=0, keepdims=True) #[:,mx.newaxis]

    
    numerator = mx.einsum('pi,lj,ij->pl', x_diff, y_diff, w)
    #numerator = y_diff@(w@x_diff.T)
    denominator = mx.einsum('i,j->ij',mx.sqrt(mx.sum(x_diff**2,axis=1)),mx.sqrt(mx.sum(y_diff**2,axis=1)))
    
    moranR = numerator / denominator

    var= (n**2*mx.einsum('ij,ji',w,w)-2*n*mx.einsum('i,i',mx.sum(w,axis=1),mx.sum(w,axis=0))+mx.einsum('ij->',w)**2)/(n**2*(n-1)**2)

    z = moranR/(var**(1/2))
    p = scipy.stats.norm.sf(z)

    return moranR,p

## Protein-Lipid Adjacency Matrix

In [ ]:
xys = []
for i in range(n_slice):
    xy = dfs[i]['spatial'].values
    xy = np.vstack(xy)
    xys.append(xy)

In [ ]:
corr_p_l = []
p_p_l = []
for i in range(n_slice):
    w = moranR_weights(xys[i],l=0.5)
    corr,p_val=moranR(proteins[i],lipids[i],w)
    corr_p_l.append(corr)
    p_p_l.append(p_val)

In [ ]:
corr_p_l = np.array(corr_p_l)
p_p_l = np.array(p_p_l)

In [ ]:
for i in range(n_protein):
    for j in range(n_lipid):
        p_p_l[:,i,j] = scipy.stats.false_discovery_control(p_p_l[:,i,j])

In [ ]:
significant = (p_p_l <= 0.05)
significant_counts = np.sum(significant, axis=0)
significant_counts

In [ ]:
avg = np.mean(significant_counts)
avg = np.ceil(avg)
avg

In [ ]:
p_l_weight = np.mean(corr_p_l,axis=0)
p_l_weight = np.where(p_l_weight >0,p_l_weight,0)
p_l_adj = np.where(significant_counts >=avg, p_l_weight, 0)
p_l_adj

In [ ]:
plt.figure(figsize=(4, 6))
flatten_p_weights = p_l_adj.flatten()
#sns.kdeplot(flatten_p_weights[flatten_p_weights>0], fill=True)
sns.histplot(flatten_p_weights[flatten_p_weights>0])
plt.title('Box plot of weights for proteins-lipids')
plt.xlabel('All Data', fontsize=12)
plt.ylabel('normalized weights', fontsize=12)
plt.show()

# 3.Normalize Adjacency Matrix

In [ ]:
max = p_adj.max()
min = p_adj.min()
p_nadj = (p_adj - min)/(max-min)
p_nadj

In [ ]:
max = l_adj.max()
min = l_adj.min()
l_nadj = (l_adj - min)/(max-min)
l_nadj

In [ ]:
max = p_l_adj.max()
min = p_l_adj.min()
p_l_nadj = (p_l_adj - min)/(max-min)
p_l_nadj

In [ ]:
plt.figure(figsize=(4, 6))
flatten_p_weights = p_l_nadj.flatten()
#sns.kdeplot(flatten_p_weights[flatten_p_weights>0], fill=True)
sns.histplot(flatten_p_weights[flatten_p_weights>0])
plt.title('Box plot of weights for proteins-lipids')
plt.xlabel('All Data', fontsize=12)
plt.ylabel('normalized weights', fontsize=12)
plt.show()

In [ ]:
import pickle
with open('data/spatial/protein.pkl', 'wb') as f:
    pickle.dump(p_adj, f)
with open('data/spatial/lipid.pkl','wb') as f:
    pickle.dump(l_adj, f)
with open('data/spatial/protein_lipid.pkl','wb') as f:
    pickle.dump(p_l_adj, f)